In [ ]:
import pandas as pd

from netsuite_auth import get_auth, run_suiteql

auth, BASE_URL = get_auth()

In [ ]:
# --- Schema: Discover all queryable fields on the item record type ---
# SELECT * on a single record returns exactly the fields available via SuiteQL.
# Use this to find field internal names before writing queries.

results = run_suiteql("SELECT * FROM item WHERE rownum <= 1")

if results:
    field_names = [{"fieldname": k} for k in results[0].keys() if k != "links"]
    df_all_fields = pd.DataFrame(field_names).sort_values("fieldname").reset_index(drop=True)
else:
    print("No item records found — cannot derive schema")
    df_all_fields = pd.DataFrame()

df_all_fields

In [ ]:
# --- Schema: Custom fields only (fields starting with "cust") ---
df_custom_fields = df_all_fields[df_all_fields["fieldname"].str.startswith("cust")].reset_index(drop=True)
df_custom_fields

In [ ]:
# --- Schema: Search for a field by label keyword ---
# Useful when you know the UI label but not the internal field name.

keyword = "weight"  # change to whatever you're looking for

df_search = df_all_fields[df_all_fields["fieldname"].str.contains(keyword, case=False, na=False)]
df_search

In [ ]:
# --- Schema: Discover fields on any record type ---
# Change RECORD_TYPE to explore other tables: transaction, customer, vendor, account, etc.

RECORD_TYPE = "transaction"

results = run_suiteql(f"SELECT * FROM {RECORD_TYPE} WHERE rownum <= 1")

if results:
    fields = [{"fieldname": k} for k in results[0].keys() if k != "links"]
    df_fields = pd.DataFrame(fields).sort_values("fieldname").reset_index(drop=True)
else:
    print(f"No records found in {RECORD_TYPE}")
    df_fields = pd.DataFrame()

df_fields